# Тема 4. Деревья решений и ансамблевые методы

**Лабораторная работа на UCI Adult**  
Цель: построить воспроизводимое сравнение Decision Tree, Random Forest, Extra Trees и Gradient Boosting без утечки данных.

> Ячейки `TODO` должны быть дополнены студентом. Test-набор оценивается только в финальном разделе.

## 0. Среда и воспроизводимость
Запустите установку один раз при необходимости. Затем сохраните версии библиотек.

In [ ]:
# %pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn
import sys, platform, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
RANDOM_STATE = 42
print(sys.version)
print('platform:', platform.platform())
print('sklearn:', sklearn.__version__)

## 1. Загрузка данных
Источник: UCI Adult, DOI `10.24432/C5XW20`. Набор содержит 48 842 объекта и 14 признаков.

In [ ]:
from ucimlrepo import fetch_ucirepo
adult = fetch_ucirepo(id=2)
X = adult.data.features.copy()
y_raw = adult.data.targets.iloc[:, 0].astype(str)
y = y_raw.str.replace('.', '', regex=False).str.strip().eq('>50K').astype(int)
print(X.shape, y.mean())
X.head()

### TODO 1 — паспорт данных
- Покажите типы, пропуски, дубликаты и распределение цели.
- Объясните, почему `sex` и `race` нельзя бездумно трактовать как причинные факторы.
- Зафиксируйте 3 потенциальных риска качества данных.

In [ ]:
# TODO: реализуйте сводку
summary = pd.DataFrame({
    'dtype': X.dtypes.astype(str),
    'missing': X.isna().sum(),
    'n_unique': X.nunique(dropna=False)
})
display(summary)
# TODO: дубликаты и график цели

## 2. Train/test и preprocessing
Test откладывается до завершения выбора модели. Все преобразования обучаются внутри Pipeline.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### TODO 2 — leakage checklist
Опишите, какие операции будут ошибочными до split. Объясните, почему `handle_unknown='ignore'` нужен для новых категорий.

## 3. Baselines

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

scoring = {'roc_auc':'roc_auc', 'pr_auc':'average_precision',
           'bal_acc':'balanced_accuracy', 'neg_log_loss':'neg_log_loss'}
models = {
    'dummy': DummyClassifier(strategy='prior'),
    'logreg': LogisticRegression(max_iter=1000, class_weight=None)
}
cv_rows = []
for name, est in models.items():
    pipe = Pipeline([('prep', prep), ('model', est)])
    out = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring,
                         n_jobs=-1, return_train_score=True)
    cv_rows.append({'model':name, **{m:out[f'test_{m}'].mean() for m in scoring},
                    'fit_s':out['fit_time'].mean()})
pd.DataFrame(cv_rows)

## 4. Короткое дерево и правила

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

tree_pipe = Pipeline([
    ('prep', prep),
    ('model', DecisionTreeClassifier(max_depth=4, min_samples_leaf=50,
                                     random_state=RANDOM_STATE))
])
tree_pipe.fit(X_train, y_train)
feature_names = tree_pipe.named_steps['prep'].get_feature_names_out()
print(export_text(tree_pipe.named_steps['model'], feature_names=list(feature_names), max_depth=4))
plt.figure(figsize=(22, 10))
plot_tree(tree_pipe.named_steps['model'], feature_names=feature_names,
          class_names=['<=50K','>50K'], filled=True, max_depth=3, fontsize=7)
plt.show()

### TODO 3 — интерпретация дерева
Выберите три листа. Запишите путь как правило и укажите: число объектов, вероятность класса 1, возможные прокси/этические риски.

## 5. Кривая сложности

In [ ]:
from sklearn.model_selection import cross_validate
records=[]
for depth in range(1, 21):
    pipe=Pipeline([('prep',prep),('model',DecisionTreeClassifier(
        max_depth=depth, min_samples_leaf=20, random_state=RANDOM_STATE))])
    s=cross_validate(pipe,X_train,y_train,cv=cv,scoring='roc_auc',
                     return_train_score=True,n_jobs=-1)
    records.append({'depth':depth,'train':s['train_score'].mean(),
                    'cv':s['test_score'].mean(),'cv_std':s['test_score'].std()})
curve=pd.DataFrame(records)
plt.plot(curve.depth,curve.train,label='train')
plt.plot(curve.depth,curve.cv,label='CV')
plt.fill_between(curve.depth,curve.cv-curve.cv_std,curve.cv+curve.cv_std,alpha=.2)
plt.xlabel('max_depth'); plt.ylabel('ROC AUC'); plt.legend(); plt.grid(alpha=.2); plt.show()

### TODO 4 — bias–variance
Отметьте underfitting, выбранную глубину и область overfitting. Повторите для `min_samples_leaf` ∈ {1, 5, 20, 50, 200}.

## 6. Cost-complexity pruning
Подберите `ccp_alpha` внутри CV. Не используйте test.

In [ ]:
from sklearn.model_selection import GridSearchCV
prune_search = GridSearchCV(
    Pipeline([('prep',prep),('model',DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    {'model__max_depth':[5,8,None],
     'model__min_samples_leaf':[1,20,50],
     'model__ccp_alpha':[0.0,1e-5,3e-5,1e-4,3e-4,1e-3]},
    scoring='roc_auc', cv=cv, n_jobs=-1
)
# TODO: fit, вывести best_params_, best_score_, глубину и число листьев

## 7. Random Forest и OOB

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_pipe = Pipeline([
    ('prep',prep),
    ('model',RandomForestClassifier(
        n_estimators=400, max_features='sqrt', min_samples_leaf=5,
        bootstrap=True, oob_score=True, n_jobs=-1, random_state=RANDOM_STATE))
])
# TODO: оцените RF по тем же фолдам; затем fit на X_train и выведите oob_score_

### TODO 5 — стабилизация леса
Для `n_estimators` ∈ {25, 50, 100, 200, 400, 800} постройте OOB score и время обучения. Не запускайте CV для каждой точки, если ресурсы ограничены.

## 8. Extra Trees

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
extra_pipe = Pipeline([
    ('prep',prep),
    ('model',ExtraTreesClassifier(n_estimators=400, min_samples_leaf=5,
                                  max_features='sqrt', n_jobs=-1,
                                  random_state=RANDOM_STATE))
])
# TODO: одинаковая схема CV; добавьте результат в итоговую таблицу

## 9. Histogram Gradient Boosting
Для простоты OHE переводится в плотный массив. На слабом компьютере используйте подвыборку train или внешний CatBoost с нативными категориями.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

dense_prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
hgb_pipe = Pipeline([
    ('prep',dense_prep),
    ('model',HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=300, max_leaf_nodes=31,
        min_samples_leaf=30, early_stopping=True, random_state=RANDOM_STATE))
])
# TODO: RandomizedSearchCV по learning_rate, max_leaf_nodes, min_samples_leaf, l2_regularization

## 10. Финальная таблица
Только после выбора конфигураций обучите модели на всем `X_train` и один раз оцените `X_test`.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, f1_score, log_loss)

def test_metrics(fitted, name, threshold=.5):
    p=fitted.predict_proba(X_test)[:,1]
    yh=(p>=threshold).astype(int)
    return {'model':name,'roc_auc':roc_auc_score(y_test,p),
            'pr_auc':average_precision_score(y_test,p),
            'balanced_acc':balanced_accuracy_score(y_test,yh),
            'f1':f1_score(y_test,yh),'log_loss':log_loss(y_test,p)}

# TODO: final_models = {...}; fit; собрать DataFrame, добавить время fit/predict

## 11. Permutation importance

In [ ]:
from sklearn.inspection import permutation_importance
# TODO: используйте лучший fitted Pipeline и X_test
# result = permutation_importance(best_model, X_test, y_test,
#     scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
# В Pipeline permutation_importance на исходном X вернет важности исходных столбцов,
# что удобно для группировки one-hot уровней.

## 12. Порог и подгруппы

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_curve
# TODO: выберите threshold только по validation predictions (cross_val_predict).
# Затем оцените test и метрики по sex/race при достаточном размере групп.

## 13. Вывод
Заполните:
1. Лучшая модель и резервный вариант.
2. Выигрыш против Dummy и Logistic Regression.
3. Компромисс качество / время / размер / интерпретируемость.
4. Риски: leakage, drift, fairness, calibration.
5. Что мониторить после внедрения.